# Model 2 — Two-Stage Factorized SupCon on B2 ViT-B/16

**Model 2** tests whether a shared visual representation can be made more
composition-aware by training it with **factorized supervised contrastive
learning** instead of crop/disease cross-entropy alone. The model is intentionally
close to Baseline 2: the same ViT-B/16 backbone, the same 55-crop / 175-disease
label space, and the same composition-scoring rule. The difference is *how the
backbone is shaped before the linear probes are trained*. By default the notebook
warm-starts the backbone from the supplied Baseline 2 ViT `best.pt` (B2 heads are
discarded); if that checkpoint is absent, it falls back to ImageNet weights.

## Motivation

Baseline 2 shares one backbone between crop and disease heads and optimizes both
with cross-entropy. That objective can reward shortcut features: a model may
predict a disease from the crop identity without learning disease evidence that
generalizes to a new crop–disease pair. Model 2 attacks this by pulling images
with the same crop into a compact crop cluster and images with the same disease
into a compact disease cluster, using two separate projection heads. The linear
heads are then trained on the frozen representation. If the contrastive stage
learns useful disease structure, the unseen-disease accuracy should improve
without collapsing seen-crop accuracy.

## Pipeline

1. **Stage 1 — factorized SupCon.** A shared timm ViT-B/16 produces a 768-d
   feature. Two non-linear projectors map that feature to 128-d crop and disease
   spaces. Each space is trained with supervised contrastive loss:
   $\mathcal{L}_{stage1} = \mathcal{L}_{SupCon}^{crop} + \lambda \mathcal{L}_{SupCon}^{disease}$,
   with $\lambda = 1.5$ to give disease structure slightly more weight.
2. **Stage 2 — frozen linear probing.** The backbone and both projectors are
   frozen. Two linear heads are trained with crop cross-entropy + disease
   cross-entropy. This isolates the quality of the Stage-1 representation and
   keeps the second stage cheap.
3. **Evaluation — GZSL protocol.** For an image, the composition score is the
   sum of crop and disease log-probabilities plus a calibrated novelty bias for
   unseen compositions:
   $s(x,c) = \log p_{crop}(x) + \log p_{disease}(x) + b \cdot \mathbf{1}[(crop,disease) \notin \mathcal{C}_{seen}]$.
   The bias is selected on a pseudo-unseen dev split, never on the test set.

## Relationship to the other baselines

- **Baseline 2 ViT** uses the same shared-backbone idea but trains with
  cross-entropy only. Model 2 is a representation-learning change, not a new
  evaluation protocol.
- **Baseline 1 ViT** uses two independent networks and is not a clean ablation
  against Model 2 because the backbone and parameter count differ.
- **Model 5** splits the 768-d feature into two 384-d subspaces and adds a
  cosine orthogonality penalty. Model 2 keeps the full 768-d feature for both
  heads, uses two projection heads, and has **no orthogonality penalty**. The
  two models should not be described as the same method.

## How to run

1. Keep `smoke=False`, `run_mode='dev'`, and `auto_final=True`, then press
   **Run All**. Dev calibrates, then the notebook automatically switches to
   final and continues in the same pass. The setup cell auto-discovers the B2
   `best.pt`; confirm the model log says `missing 0 | unexpected 0`.
2. To stop after dev, set `auto_final=False`. Then inspect
   `runs/dev_summary.json` before starting final.
3. To skip dev and run final only, set `run_mode='final'`; this requires an
   existing `runs/dev_summary.json`.
4. The **Visual diagnostics** cell runs automatically after final.

Set `CFG['smoke'] = True` for a fast 1-epoch subset check. Smoke outputs go to
`runs_smoke/` and never overwrite real results, and smoke never auto-switches to
final. The notebook is local-first: it reads `DPD_Small/` and
`.runtime/dpd_small_320/`, and writes under `DeepLearning/Model 2/runs/`.

In [ ]:
import os, sys, json, math, random, time, subprocess, importlib.util, hashlib
from pathlib import Path

def find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / 'DPD_Small').exists() and (p / 'Baseline1').exists():
            return p
    raise FileNotFoundError('Could not locate repository root containing DPD_Small and Baseline1')

REPO_ROOT = find_repo_root(Path.cwd())
MODEL_DIR = REPO_ROOT / 'DeepLearning' / 'Model 2'
DATA_DIR = REPO_ROOT / 'DPD_Small'
CACHE_DIR = REPO_ROOT / '.runtime' / 'dpd_small_320'
IMAGE_DIR = CACHE_DIR / 'DPD_Small_dataset'

os.environ.setdefault('TORCH_HOME', str(REPO_ROOT / '.runtime' / 'torch'))
os.environ.setdefault('HF_HOME', str(REPO_ROOT / '.runtime' / 'huggingface'))

if importlib.util.find_spec('timm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'timm'], check=True)
if importlib.util.find_spec('tqdm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'tqdm'], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
import timm

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEVICE == 'cuda'
print('device:', DEVICE, torch.cuda.get_device_name(0) if AMP else '(CPU)')

assert DATA_DIR.exists(), f'Missing dataset directory: {DATA_DIR}'
assert IMAGE_DIR.exists(), f'Missing image cache: {IMAGE_DIR}'

CFG = dict(
    run_mode='dev',                 # 'dev' then 'final'
    auto_final=True,                # after dev, switch to final in the same Run All
    smoke=False,                    # True = 1-epoch subset smoke test
    b2_init_ckpt=None,              # auto-discovered below; set manually to override
    backbone='vit_base_patch16_224',
    pretrained=True,                # ImageNet fallback; disabled when B2 init is found
    img_size=224,
    stage1_epochs=10,
    stage2_epochs=10,
    stage1_batch_size=8,            # lower to 4 if the 6 GB GPU OOMs
    stage2_batch_size=16,
    eval_batch_size=16,
    num_workers=0,
    lr_backbone=3e-5,
    lr_projection=3e-4,
    lr_head=1e-3,
    weight_decay=0.05,
    head_weight_decay=1e-4,
    supcon_temp=0.07,
    lambda_disease=1.5,
    label_smoothing=0.1,
    proj_hidden=256,
    proj_dim=128,
    drop_rate=0.0,
    num_holdout_pairs=15,
    val_frac=0.1,
    seed=42,
)

_b2_candidates = sorted((MODEL_DIR / 'B2_ViT(main DL)').glob('**/baseline2_multitask_vit/final/best.pt'))
if CFG['b2_init_ckpt'] is None and _b2_candidates:
    CFG['b2_init_ckpt'] = str(_b2_candidates[0])
if CFG['b2_init_ckpt']:
    CFG['pretrained'] = False
    assert Path(CFG['b2_init_ckpt']).exists(), f"B2 checkpoint not found: {CFG['b2_init_ckpt']}"

print('B2 init:', CFG['b2_init_ckpt'] if CFG['b2_init_ckpt'] else 'none (ImageNet fallback)')

assert CFG['run_mode'] in ('dev', 'final')
if CFG['smoke']:
    assert CFG['run_mode'] == 'dev', 'smoke mode is dev-only'
    CFG.update(
        stage1_epochs=1,
        stage2_epochs=1,
        stage1_batch_size=4,
        stage2_batch_size=4,
        num_holdout_pairs=3,
    )

OUT_DIR = MODEL_DIR / ('runs_smoke' if CFG['smoke'] else 'runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEV_SUMMARY = OUT_DIR / 'dev_summary.json'

seed_everything(CFG['seed'])
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()}, indent=2))
print('out_dir:', OUT_DIR)

## Metadata and composition mappings

The DPD metadata CSVs have no header and seven columns:
`file, crop_id, disease_id, composition_id, crop_name, disease_name, composition_name`.
The notebook rebuilds the three label spaces directly from the data so that the
55 crop classes, 175 disease classes, and 333 seen compositions are aligned with
the image folders and with the Baseline 2 checkpoints.

A **composition** is a specific crop–disease pair. The training set contains 323
seen compositions; the unseen test set contains 10 held-out compositions. This is
the core difficulty of the assignment: both the crop and the disease may have
been seen during training, but their *combination* has not. The mappings
`COMP_CROP` and `COMP_DIS` convert a composition index into its crop/disease
indices, and `NOVEL_MASK` marks which of the 333 compositions are unseen. The
final evaluation builds `NOVEL_MASK` from `train_df`, so no test label leaks into
calibration.

In [ ]:
COLS = ['file', 'crop', 'disease', 'comp', 'crop_name', 'disease_name', 'comp_name']

def read_split(path):
    return pd.read_csv(path, header=None, names=COLS)

train_df = read_split(DATA_DIR / 'DPD_Small_train.csv')
test_seen_df = read_split(DATA_DIR / 'DPD_Small_test_seen.csv')
test_unseen_df = read_split(DATA_DIR / 'DPD_Small_test_unseen.csv')
all_df = pd.concat([train_df, test_seen_df, test_unseen_df], ignore_index=True)

assert (all_df.groupby('comp')[['crop', 'disease']].nunique() == 1).all().all()
comps = (all_df.drop_duplicates('comp').sort_values('comp')
         [['comp', 'crop', 'disease', 'crop_name', 'disease_name', 'comp_name']]
         .reset_index(drop=True))
assert comps['comp'].tolist() == list(range(len(comps)))

NUM_CROPS = int(all_df['crop'].max()) + 1
NUM_DISEASES = int(all_df['disease'].max()) + 1
NUM_COMPS = len(comps)
COMP_CROP = torch.tensor(comps['crop'].values)
COMP_DIS = torch.tensor(comps['disease'].values)
COMP_NAMES = comps['comp_name'].tolist()
CROP_NAMES = all_df.drop_duplicates('crop').set_index('crop')['crop_name'].sort_index().tolist()
DISEASE_NAMES = all_df.drop_duplicates('disease').set_index('disease')['disease_name'].sort_index().tolist()

TRAIN_COMPS = set(train_df['comp'])
UNSEEN_COMPS = set(test_unseen_df['comp'])
assert not TRAIN_COMPS & UNSEEN_COMPS

hasher = hashlib.sha256()
for path in (DATA_DIR / 'DPD_Small_train.csv', DATA_DIR / 'DPD_Small_test_seen.csv', DATA_DIR / 'DPD_Small_test_unseen.csv'):
    with open(path, 'rb') as f:
        hasher.update(f.read())
data_fingerprint = hasher.hexdigest()

print(f'train {len(train_df)} | test-seen {len(test_seen_df)} | test-unseen {len(test_unseen_df)}')
print(f'crops {NUM_CROPS} | diseases {NUM_DISEASES} | compositions {NUM_COMPS} '
      f'(seen {len(TRAIN_COMPS)}, unseen {len(UNSEEN_COMPS)})')
print('data fingerprint:', data_fingerprint[:16])

## Pseudo-unseen dev split

Test-unseen labels cannot be used to choose an epoch or a novelty bias, so the
notebook creates a **pseudo-unseen** validation problem from the training set.
Fifteen seen compositions are held out as `val_unseen`; 10% of the remaining
images become `val_seen`; the rest is `fit_df`. Stage 1 and Stage 2 fit only on
`fit_df`. This mirrors the real evaluation: the model sees the individual crops
and diseases during training, but not the held-out composition pairs.

The split is controlled by `CFG['dev_holdout_pairs']`, `CFG['val_frac']`, and
`CFG['seed']`. The same seed is used every run, so the dev comparison is stable.
`val_seen` and `val_unseen` are used for epoch selection and bias calibration;
the final run does **not** use them for training. The test set is untouched until
the final evaluation cell.

**Important:** the held-out pairs are unseen *compositions*, not unseen crops or
unseen diseases. This matches the assignment's unseen-composition setting and is
why the 10% dev-seen split still covers the same crop/disease classes.

In [ ]:
def make_dev_split(df, k, val_frac, seed):
    rng = np.random.RandomState(seed)
    pairs = df.groupby('comp').agg(crop=('crop', 'first'), disease=('disease', 'first'))
    remaining, holdout = pairs.copy(), []
    for c in rng.permutation(pairs.index.values):
        if len(holdout) == k:
            break
        rest = remaining.drop(c)
        if (rest['crop'] == pairs.at[c, 'crop']).any() and (rest['disease'] == pairs.at[c, 'disease']).any():
            holdout.append(int(c))
            remaining = rest
    val_unseen = df[df['comp'].isin(holdout)]
    rest_df = df[~df['comp'].isin(holdout)]
    val_idx = []
    for _, g in rest_df.groupby('comp'):
        n_val = int(len(g) * val_frac)
        if n_val:
            val_idx += g.sample(n_val, random_state=seed).index.tolist()
    val_seen = rest_df.loc[val_idx]
    fit = rest_df.drop(index=val_idx)
    return fit.reset_index(drop=True), val_seen.reset_index(drop=True), val_unseen.reset_index(drop=True), sorted(holdout)

if CFG['run_mode'] == 'dev':
    fit_df, val_seen_df, val_unseen_df, HOLDOUT = make_dev_split(
        train_df, CFG['num_holdout_pairs'], CFG['val_frac'], CFG['seed']
    )
else:
    fit_df, val_seen_df, val_unseen_df, HOLDOUT = train_df.copy(), None, None, []

if CFG['smoke']:
    fit_df = fit_df.sample(n=min(256, len(fit_df)), random_state=CFG['seed']).reset_index(drop=True)
    val_seen_df = val_seen_df.sample(n=min(64, len(val_seen_df)), random_state=CFG['seed']).reset_index(drop=True)
    val_unseen_df = val_unseen_df.sample(n=min(64, len(val_unseen_df)), random_state=CFG['seed']).reset_index(drop=True)

assert len(fit_df) > 0
if CFG['run_mode'] == 'dev':
    assert len(val_seen_df) > 0 and len(val_unseen_df) > 0, 'Dev split must contain seen and pseudo-unseen images'

FIT_COMPS = sorted(set(fit_df['comp']))
NOVEL_MASK = torch.ones(NUM_COMPS, dtype=torch.bool)
NOVEL_MASK[FIT_COMPS] = False

print(f'fit {len(fit_df)} | val-seen {0 if val_seen_df is None else len(val_seen_df)} | '
      f'val-unseen {0 if val_unseen_df is None else len(val_unseen_df)} | holdout {len(HOLDOUT)}')
print('holdout pairs:', [COMP_NAMES[c] for c in HOLDOUT])

## Two-view transforms and loaders

Supervised contrastive learning needs at least two views of the same image so
that the loss can pull the two views together while pushing other samples away.
Stage 1 therefore uses a **two-view transform**: two independent random crops
(scale 0.6–1.0), horizontal and vertical flips, mild rotation, color jitter, and
ImageNet normalization. The two views are returned as a tuple; each image
contributes two 224×224 tensors to the contrastive batch.

Stage 2 uses a single-view B2-style augmentation because it only trains linear
heads. Evaluation is deterministic and matches B2 v2: resize to 224×224 with
bilinear interpolation, then normalize. The loader uses `drop_last=True` in
Stage 1 so every contrastive batch has complete positive sets; Stage 2 and
evaluation keep all images.

`CFG['stage1_batch_size']=8` means 8 source images × 2 views = 16 ViT inputs per
step. On a 6 GB laptop GPU this is the main memory risk; lower it to 4 if
PyTorch reports an out-of-memory error. Gradient accumulation would not enlarge
the SupCon positive set, so it is not used as a substitute for batch size.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
S = CFG['img_size']

supcon_tf = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.6, 1.0), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20, fill=128),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

stage2_tf = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.8, 1.0), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15, fill=128),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((S, S), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class TwoViewTransform:
    def __init__(self, base_tf):
        self.base_tf = base_tf

    def __call__(self, img):
        return self.base_tf(img), self.base_tf(img)

class DPDSmallDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.files = df['file'].tolist()
        self.labels = df[['crop', 'disease', 'comp']].to_numpy()
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        with Image.open(os.path.join(self.img_dir, self.files[i])) as im:
            image = im.convert('RGB')
        crop, disease, comp = self.labels[i]
        return self.transform(image), int(crop), int(disease), int(comp)

def make_loader(df, transform, batch_size, shuffle=False, drop_last=False):
    ds = DPDSmallDataset(df, IMAGE_DIR, transform)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=CFG['num_workers'],
        pin_memory=AMP,
        persistent_workers=CFG['num_workers'] > 0,
    )

## Factorized SupCon model

The model object contains four parts:

- `backbone` — timm `vit_base_patch16_224`, warm-started from the Baseline 2 ViT
  checkpoint when available (otherwise ImageNet), returning a pooled 768-d
  feature $f(x)$.
- `crop_projector` and `disease_projector` — Stage-1-only MLPs
  $g_c, g_d: \mathbb{R}^{768} \rightarrow \mathbb{R}^{128}$, each ending in an
  L2-normalized embedding.
- `head_crop` and `head_disease` — Stage-2-only linear classifiers on the same
  768-d feature.

For a pair of views $(x_i, x_j)$ of the same image, Stage 1 computes
$z_i^c = g_c(f(x_i))$, $z_j^c = g_c(f(x_j))$ and the analogous disease
embeddings. The supervised contrastive objective treats two views with the same
crop label as positives and all other crop labels as negatives; the disease loss
does the same in the disease space. The total Stage-1 loss is

$$
\mathcal{L}_{stage1} = \mathcal{L}_{SupCon}^{crop} + \lambda \mathcal{L}_{SupCon}^{disease},
\qquad \lambda = 1.5.
$$

The temperature is $\tau = 0.07$. A smaller temperature makes the loss focus on
the hardest negatives; 0.07 is the standard SupCon setting. The factorized
design gives crop and disease separate embedding geometries while sharing the
visual backbone, so disease learning is not forced to use the same projection as
crop learning.

Stage 2 freezes `backbone`, `crop_projector`, and `disease_projector` and trains
only the two linear heads with
$\mathcal{L}_{stage2} = CE(\hat{y}_{crop}, y_{crop}) + CE(\hat{y}_{disease}, y_{disease})$.
The full model state is saved so Stage 1 and Stage 2 checkpoints load without key
mismatches; `eval_state_dict()` returns only the inference-relevant weights.

Unlike Model 5, Model 2 does **not** split the 768-d feature into two 384-d
subspaces and does **not** add an orthogonality penalty. The factorization is in
the projection heads, not in the backbone feature itself.

In [ ]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features.float(), dim=-1)
        bsz, n_views, dim = features.shape
        device = features.device

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        contrast = torch.cat(torch.unbind(features, dim=1), dim=0)
        sim = torch.matmul(contrast, contrast.T) / self.temperature
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()

        mask = mask.repeat(n_views, n_views)
        logits_mask = 1.0 - torch.eye(bsz * n_views, device=device)
        mask = mask * logits_mask
        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True).clamp(min=1e-12))

        pos_per_row = mask.sum(dim=1).clamp(min=1.0)
        loss = -(mask * log_prob).sum(dim=1) / pos_per_row
        return loss.mean()

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

class FactorizedSupConViT(nn.Module):
    def __init__(self, backbone_name, n_crops, n_diseases, proj_hidden=256, proj_dim=128,
                 pretrained=True, drop_rate=0.0):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained, num_classes=0, drop_rate=drop_rate
        )
        d = self.backbone.num_features
        self.crop_projector = ProjectionHead(d, proj_hidden, proj_dim)
        self.disease_projector = ProjectionHead(d, proj_hidden, proj_dim)
        self.head_crop = nn.Linear(d, n_crops)
        self.head_disease = nn.Linear(d, n_diseases)

    def forward_stage1(self, x1, x2):
        feats = self.backbone(torch.cat([x1, x2], dim=0))
        f1, f2 = feats.chunk(2, dim=0)
        zc = torch.stack([
            F.normalize(self.crop_projector(f1), dim=-1),
            F.normalize(self.crop_projector(f2), dim=-1),
        ], dim=1)
        zd = torch.stack([
            F.normalize(self.disease_projector(f1), dim=-1),
            F.normalize(self.disease_projector(f2), dim=-1),
        ], dim=1)
        return zc, zd

    def forward(self, x):
        feat = self.backbone(x)
        return self.head_crop(feat), self.head_disease(feat), feat

    def eval_state_dict(self):
        keep = ('backbone.', 'head_crop.', 'head_disease.')
        return {k: v for k, v in self.state_dict().items() if k.startswith(keep)}

def maybe_init_from_b2(model, path):
    if not path:
        return
    path = Path(path)
    if not path.exists():
        print(f'B2 init checkpoint not found, using ImageNet weights: {path}')
        return
    ck = torch.load(path, map_location='cpu', weights_only=False)
    state = ck.get('model', ck)
    bb = {k[len('backbone.'):]: v for k, v in state.items() if k.startswith('backbone.')}
    if not bb:
        print('No backbone.* keys found in B2 checkpoint; using ImageNet weights.')
        return
    missing, unexpected = model.backbone.load_state_dict(bb, strict=False)
    print(f'Initialised backbone from {path} | missing {len(missing)} | unexpected {len(unexpected)}')

def build_model():
    model = FactorizedSupConViT(
        CFG['backbone'], NUM_CROPS, NUM_DISEASES,
        proj_hidden=CFG['proj_hidden'], proj_dim=CFG['proj_dim'],
        pretrained=CFG['pretrained'], drop_rate=CFG['drop_rate'],
    )
    maybe_init_from_b2(model, CFG['b2_init_ckpt'])
    model = model.to(DEVICE)
    print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M | feature dim: {model.backbone.num_features}")
    return model

## Evaluation functions

The evaluation follows the Baseline 2 v2 Generalized Zero-Shot Learning
protocol. For image $x$ and composition $c=(a,d)$:

$$
s(x,c) = \log p_{crop}(a \mid x) + \log p_{disease}(d \mid x)
          + b \cdot \mathbf{1}[c \notin \mathcal{C}_{seen}].
$$

The sum of log-probabilities is the log of the product
$p_{crop}(a \mid x)\,p_{disease}(d \mid x)$, so each composition is ranked by how
well its crop and disease parts agree. The bias $b$ is a novelty offset: because
unseen compositions never appear as training labels, their combined scores tend
to be lower, and a positive $b$ compensates for that.

`topk_acc` applies the bias and reports top-1 or top-5 composition accuracy.
`harmonic_mean` is

$$
H = \frac{2 \cdot Acc_{seen} \cdot Acc_{unseen}}{Acc_{seen} + Acc_{unseen}},
$$

which forces the model to do well on both domains instead of sacrificing one.
`bias_sweep` scans candidate biases and returns the best $H$, the corresponding
seen/unseen accuracies, and the area under the seen–unseen curve (AUC). AUC is a
bias-independent summary of how well the model separates seen and unseen pairs;
it is not the same as top-1 accuracy.

`paper_protocol` measures crop, disease, and exact crop+disease accuracy from the
individual heads. `shortcut_rate` checks whether a wrong disease prediction is
actually a disease that co-occurs with the predicted crop in the training set;
a high shortcut rate suggests the model is using crop context instead of disease
evidence.

The dev run selects $b$ on pseudo-unseen validation. The test **oracle** bias is
reported only as an upper bound; it must not be used as the headline result.

In [ ]:
@torch.no_grad()
def run_inference(model, loader):
    model.eval()
    out = {k: [] for k in ('crop_lp', 'dis_lp', 'feat', 'crop', 'disease', 'comp')}
    for x, crop, disease, comp in loader:
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
            lc, ld, feat = model(x.to(DEVICE, non_blocking=True))
        out['crop_lp'].append(F.log_softmax(lc.float(), dim=1).cpu())
        out['dis_lp'].append(F.log_softmax(ld.float(), dim=1).cpu())
        out['feat'].append(feat.float().cpu())
        out['crop'].append(crop)
        out['disease'].append(disease)
        out['comp'].append(comp)
    return {k: torch.cat(v) for k, v in out.items()}

def comp_scores(out):
    return out['crop_lp'][:, COMP_CROP] + out['dis_lp'][:, COMP_DIS]

def topk_acc(scores, labels, k=1, bias=0.0, novel_mask=None):
    novel_mask = NOVEL_MASK if novel_mask is None else novel_mask
    s = scores + bias * novel_mask.float()
    return (s.topk(k, dim=1).indices == labels[:, None]).any(dim=1).float().mean().item()

def harmonic_mean(s, u):
    return 2 * s * u / (s + u) if s + u > 0 else 0.0

def bias_sweep(seen_scores, seen_labels, novel_scores, novel_labels, max_points=200, novel_mask=None):
    novel_mask = NOVEL_MASK if novel_mask is None else novel_mask
    assert (~novel_mask).any() and novel_mask.any(), 'bias_sweep needs both seen and novel pairs'
    allsc = torch.cat([seen_scores, novel_scores])
    gap = (allsc[:, ~novel_mask].max(1).values - allsc[:, novel_mask].max(1).values).sort().values
    pick = torch.linspace(0, len(gap) - 1, min(max_points, len(gap))).long()
    biases = torch.cat([gap[:1] - 1.0, gap[pick] + 1e-4, torch.zeros(1)]).unique().tolist()
    seen = [topk_acc(seen_scores, seen_labels, 1, b, novel_mask) for b in biases]
    novel = [topk_acc(novel_scores, novel_labels, 1, b, novel_mask) for b in biases]
    hm = [harmonic_mean(s, u) for s, u in zip(seen, novel)]
    best = int(np.argmax(hm))
    order = np.argsort(seen)
    trapezoid = getattr(np, 'trapezoid', None) or np.trapz
    auc = float(trapezoid(np.array(novel)[order], np.array(seen)[order]))
    return dict(biases=biases, seen=seen, novel=novel, hm=hm, auc=auc,
                best_bias=biases[best], best_hm=hm[best],
                best_seen=seen[best], best_novel=novel[best])

def paper_protocol(out):
    pc, pdis = out['crop_lp'].argmax(1), out['dis_lp'].argmax(1)
    return {
        'crop acc': (pc == out['crop']).float().mean().item(),
        'disease acc': (pdis == out['disease']).float().mean().item(),
        'crop+disease acc': ((pc == out['crop']) & (pdis == out['disease'])).float().mean().item(),
    }

def shortcut_rate(out):
    train_pairs = set(zip(fit_df['crop'], fit_df['disease']))
    pred_dis = out['dis_lp'].argmax(1)
    wrong = pred_dis != out['disease']
    flags = [(int(c), int(d)) in train_pairs for c, d in zip(out['crop'][wrong], pred_dis[wrong])]
    return float(np.mean(flags)) if flags else 0.0

## Training functions

This cell defines the full two-stage training loop and exposes batch-level
`tqdm` progress. Stage 1 uses AdamW with a lower learning rate on the pretrained
backbone (`3e-5`) and a higher rate on the projection heads (`3e-4`), a one-epoch
linear warmup, cosine decay, gradient clipping at 1.0, and automatic mixed
precision. The backbone is fully fine-tuned; it is not frozen in Stage 1. Each
epoch writes `history_stage1_<tag>.csv` and a resumable
`stage1_<tag>_last.pt` checkpoint.

Stage 2 freezes the backbone and both projectors, switches them to eval mode,
and trains only `head_crop` and `head_disease` with cross-entropy. The learning
rate is `1e-3`, the loss is crop CE + disease CE, and the scheduler is cosine
annealing. On the dev split, every epoch runs inference on `val_seen` and
`val_unseen`, builds the bias sweep, and saves the best pseudo-unseen-HM
checkpoint as `stage2_dev_best.pt`. The final run skips validation and trains for
the epoch count selected by dev, saving `stage2_final_last.pt`.

The design choices are deliberate: Stage 1 learns the representation that the
linear probes will use, while Stage 2 keeps the comparison with Baseline 2 fair
and cheap. The notebook uses a single seed (`42`) and no class-balanced sampler;
Model 5 owns the sampler/orthogonality experiment. If the 6 GB GPU runs out of
memory, reduce `CFG['stage1_batch_size']` to 4 before rerunning.

In [ ]:
def stage1_lr_lambda(step, warmup_steps, total_steps):
    if step < warmup_steps:
        return (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

def train_stage1(fit_df, tag, epochs=None, batch_size=None):
    epochs = CFG['stage1_epochs'] if epochs is None else epochs
    batch_size = CFG['stage1_batch_size'] if batch_size is None else batch_size
    loader = make_loader(fit_df, TwoViewTransform(supcon_tf), batch_size, shuffle=True, drop_last=True)
    if len(loader) == 0:
        raise ValueError('Stage 1 loader is empty; reduce batch size or increase data')

    model = build_model()
    supcon = SupConLoss(temperature=CFG['supcon_temp'])
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': CFG['lr_backbone'], 'weight_decay': CFG['weight_decay']},
        {'params': list(model.crop_projector.parameters()) + list(model.disease_projector.parameters()),
         'lr': CFG['lr_projection'], 'weight_decay': CFG['head_weight_decay']},
    ])
    steps_per_epoch = len(loader)
    warmup_steps = max(1, steps_per_epoch)
    total_steps = epochs * steps_per_epoch
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: stage1_lr_lambda(step, warmup_steps, total_steps)
    )
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    trainable = [p for group in optimizer.param_groups for p in group['params']]
    history = []

    print(f'[stage1:{tag}] starting {epochs} epochs on {len(fit_df)} images', flush=True)
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = loss_c_sum = loss_d_sum = 0.0
        n = 0
        pbar = tqdm(loader, desc=f'stage1:{tag} {epoch}/{epochs}', leave=False, dynamic_ncols=True)
        for (x1, x2), crop, disease, _ in pbar:
            x1 = x1.to(DEVICE, non_blocking=True)
            x2 = x2.to(DEVICE, non_blocking=True)
            crop = crop.to(DEVICE, non_blocking=True)
            disease = disease.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
                zc, zd = model.forward_stage1(x1, x2)
                loss_c = supcon(zc, crop)
                loss_d = supcon(zd, disease)
                loss = loss_c + CFG['lambda_disease'] * loss_d

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            loss_sum += loss.item() * x1.size(0)
            loss_c_sum += loss_c.item() * x1.size(0)
            loss_d_sum += loss_d.item() * x1.size(0)
            n += x1.size(0)
            pbar.set_postfix(loss=f'{loss.item():.4f}', crop=f'{loss_c.item():.3f}',
                             disease=f'{loss_d.item():.3f}')

        row = dict(
            epoch=epoch,
            train_loss=loss_sum / n,
            crop_supcon=loss_c_sum / n,
            disease_supcon=loss_d_sum / n,
            lr=optimizer.param_groups[0]['lr'],
        )
        history.append(row)
        pd.DataFrame(history).to_csv(OUT_DIR / f'history_stage1_{tag}.csv', index=False)
        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'epoch': epoch,
            'history': history,
            'cfg': {k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()},
            'data_fingerprint': data_fingerprint,
            'variant': 'model2_factorized_supcon',
        }, OUT_DIR / f'stage1_{tag}_last.pt')
        print(f'[stage1:{tag}] epoch {epoch:02d}/{epochs} | loss {row["train_loss"]:.4f} '
              f'| crop {row["crop_supcon"]:.4f} | disease {row["disease_supcon"]:.4f}', flush=True)

    return model, history

def train_stage2(model, fit_df, tag, epochs=None, batch_size=None, val_seen_df=None, val_unseen_df=None, known_bias=0.0):
    epochs = CFG['stage2_epochs'] if epochs is None else epochs
    batch_size = CFG['stage2_batch_size'] if batch_size is None else batch_size
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in list(model.crop_projector.parameters()) + list(model.disease_projector.parameters()):
        p.requires_grad = False

    loader = make_loader(fit_df, stage2_tf, batch_size, shuffle=True, drop_last=False)
    head_params = list(model.head_crop.parameters()) + list(model.head_disease.parameters())
    optimizer = torch.optim.AdamW(head_params, lr=CFG['lr_head'], weight_decay=CFG['head_weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs), eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    history = []
    best_hm = -1.0
    best_epoch = 0
    best_bias = 0.0
    checkpoint_bias = float(known_bias)

    print(f'[stage2:{tag}] starting {epochs} epochs on {len(fit_df)} images', flush=True)
    for epoch in range(1, epochs + 1):
        model.eval()
        model.head_crop.train()
        model.head_disease.train()
        loss_sum = crop_hit = dis_hit = n = 0
        pbar = tqdm(loader, desc=f'stage2:{tag} {epoch}/{epochs}', leave=False, dynamic_ncols=True)
        for x, crop, disease, _ in pbar:
            x = x.to(DEVICE, non_blocking=True)
            crop = crop.to(DEVICE, non_blocking=True)
            disease = disease.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
                lc, ld, _ = model(x)
                loss = criterion(lc, crop) + criterion(ld, disease)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.item() * x.size(0)
            crop_hit += (lc.argmax(1) == crop).sum().item()
            dis_hit += (ld.argmax(1) == disease).sum().item()
            n += x.size(0)
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        scheduler.step()
        row = dict(epoch=epoch, train_loss=loss_sum / n, train_crop_acc=crop_hit / n,
                   train_disease_acc=dis_hit / n, lr=optimizer.param_groups[0]['lr'])

        if val_seen_df is not None and val_unseen_df is not None:
            print(f'[stage2:{tag}] validating epoch {epoch:02d}...', flush=True)
            vs = run_inference(model, make_loader(val_seen_df, eval_tf, CFG['eval_batch_size']))
            vu = run_inference(model, make_loader(val_unseen_df, eval_tf, CFG['eval_batch_size']))
            S_val, U_val = comp_scores(vs), comp_scores(vu)
            sweep = bias_sweep(S_val, vs['comp'], U_val, vu['comp'])
            row.update(
                val_seen_top1_b0=topk_acc(S_val, vs['comp'], 1, 0.0),
                val_unseen_top1_b0=topk_acc(U_val, vu['comp'], 1, 0.0),
                val_best_hm=sweep['best_hm'],
                val_best_bias=sweep['best_bias'],
                val_auc=sweep['auc'],
            )
            if sweep['best_hm'] > best_hm:
                best_hm = float(sweep['best_hm'])
                best_epoch = epoch
                best_bias = float(sweep['best_bias'])
                checkpoint_bias = best_bias
                torch.save({
                    'model': model.state_dict(),
                    'eval_state_dict': model.eval_state_dict(),
                    'epoch': epoch,
                    'dev_bias': best_bias,
                    'val_hm': best_hm,
                    'val_auc': float(sweep['auc']),
                    'cfg': {k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()},
                    'data_fingerprint': data_fingerprint,
                    'variant': 'model2_factorized_supcon',
                }, OUT_DIR / f'stage2_{tag}_best.pt')
        else:
            best_epoch = epoch

        history.append(row)
        pd.DataFrame(history).to_csv(OUT_DIR / f'history_stage2_{tag}.csv', index=False)
        torch.save({
            'model': model.state_dict(),
            'eval_state_dict': model.eval_state_dict(),
            'epoch': epoch,
            'dev_bias': checkpoint_bias,
            'history': history,
            'cfg': {k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()},
            'data_fingerprint': data_fingerprint,
            'variant': 'model2_factorized_supcon',
        }, OUT_DIR / f'stage2_{tag}_last.pt')
        msg = f'[stage2:{tag}] epoch {epoch:02d}/{epochs} | CE {row["train_loss"]:.4f}'
        if 'val_best_hm' in row:
            msg += f' | val HM {row["val_best_hm"]:.4f} | bias {row["val_best_bias"]:.3f}'
        print(msg, flush=True)

    return model, history, dict(best_epoch=best_epoch, best_hm=best_hm, best_bias=best_bias)

## Dev run

The dev run is the calibration phase. It trains Stage 1 and Stage 2 only on
`fit_df`, evaluates every Stage-2 epoch on the pseudo-unseen pair split, and
selects the epoch with the best harmonic mean. It then reloads that best
checkpoint and writes:

- `runs/dev_summary.json` — best Stage-2 epoch, dev bias, dev seen/unseen
  accuracies, AUC, holdout pair names, and the full configuration.
- `runs/stage1_dev_last.pt` — final Stage-1 checkpoint.
- `runs/stage2_dev_best.pt` — best pseudo-unseen checkpoint.
- `runs/stage2_dev_last.pt` — last Stage-2 checkpoint.
- `runs/history_stage1_dev.csv`, `runs/history_stage2_dev.csv`.
- `runs/dev_bias_curve.png` — seen vs unseen accuracy as the novelty bias moves.

Use the dev curve to check that the bias is doing something sensible: a good
curve should rise from low unseen accuracy at $b=0$ toward a balanced point, then
eventually trade seen accuracy away. The selected point is the maximum-H point,
not the maximum-AUC point. The dev HM is an internal model-selection number; the
final test table is the reportable result. If `run_mode='final'`, this cell
skips. When `auto_final=True` and `smoke=False`, this cell flips
`CFG['run_mode']` to `'final'` at the end, so one **Run All** continues directly
into the final cell. Set `auto_final=False` to stop after dev.

In [ ]:
if CFG['run_mode'] != 'dev':
    print('Skipping dev run. Set CFG["run_mode"] = "dev" to calibrate.')
else:
    torch.cuda.empty_cache()
    dev_model, s1_history = train_stage1(fit_df, tag='dev')
    dev_model, s2_history, best = train_stage2(
        dev_model, fit_df, tag='dev',
        val_seen_df=val_seen_df, val_unseen_df=val_unseen_df,
    )

    best_path = OUT_DIR / 'stage2_dev_best.pt'
    if not best_path.exists():
        raise FileNotFoundError(f'No dev checkpoint written: {best_path}')
    best_ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    dev_model.load_state_dict(best_ckpt['model'])

    vs = run_inference(dev_model, make_loader(val_seen_df, eval_tf, CFG['eval_batch_size']))
    vu = run_inference(dev_model, make_loader(val_unseen_df, eval_tf, CFG['eval_batch_size']))
    S_dev, U_dev = comp_scores(vs), comp_scores(vu)
    sweep = bias_sweep(S_dev, vs['comp'], U_dev, vu['comp'])

    summary = dict(
        variant='model2_factorized_supcon',
        stage1_epochs=CFG['stage1_epochs'],
        stage2_best_epoch=int(best_ckpt['epoch']),
        best_bias=float(sweep['best_bias']),
        val_best_hm=float(sweep['best_hm']),
        val_seen_at_best=float(sweep['best_seen']),
        val_unseen_at_best=float(sweep['best_novel']),
        val_auc=float(sweep['auc']),
        holdout_pairs=[COMP_NAMES[c] for c in HOLDOUT],
        data_fingerprint=data_fingerprint,
        cfg={k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()},
    )
    with open(DEV_SUMMARY, 'w') as f:
        json.dump(summary, f, indent=2)
    print(json.dumps({k: v for k, v in summary.items() if k != 'cfg'}, indent=2))

    plt.figure(figsize=(4.5, 4))
    plt.plot(sweep['seen'], sweep['novel'], '-')
    plt.scatter([sweep['best_seen']], [sweep['best_novel']], c='r', zorder=3,
                label=f'best H={sweep["best_hm"]:.3f}')
    plt.xlabel('dev-seen accuracy')
    plt.ylabel('dev-unseen accuracy')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title(f'Model 2 dev bias sweep (AUC {sweep["auc"]:.3f})')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'dev_bias_curve.png', dpi=150)
    plt.show()

    if CFG['auto_final'] and not CFG['smoke']:
        CFG['run_mode'] = 'final'
        print('Dev complete: auto-switching to final for the next cell.', flush=True)

## Final run and test evaluation

This cell is the final, reportable run. It only executes when
`CFG['run_mode'] = 'final'`; with `auto_final=True`, the dev cell switches to
final automatically at the end of the same **Run All**. It reads
`runs/dev_summary.json`, takes the dev-selected Stage-2 epoch count and novelty
bias, retrains Stage 1 on **all 8,000 training images**, and retrains Stage 2 for
the selected number of epochs.
No test image or test label is used for training or calibration.

The evaluation reports three rows:

1. **$b=0$** — no novelty calibration; this shows the raw seen/unseen trade-off.
2. **dev bias (REPORTED)** — the headline result, using the bias selected on
   pseudo-unseen dev data.
3. **test oracle** — the best possible bias on the test set; this is an upper
   bound only, not a deployable result.

It also reports seen/unseen AUC, crop/disease/exact-pair head accuracy, the
paper-protocol harmonic mean, and the shortcut rate. The final artifacts are
`runs/metrics.json`, `runs/test_bias_curve.png`, and the final checkpoints.
Compare this table against Baseline 2 ViT; do not compare the dev HM directly
with Baseline 1's dev HM because the backbone and protocol differ.

In [ ]:
if CFG['run_mode'] != 'final':
    print('Skipping final run. Set CFG["run_mode"] = "final" after reviewing dev_summary.json.')
else:
    if not DEV_SUMMARY.exists():
        raise FileNotFoundError(f'Missing dev summary: {DEV_SUMMARY}. Run dev mode first.')
    dev = json.load(open(DEV_SUMMARY))
    stage2_epochs = max(1, int(dev['stage2_best_epoch']))
    dev_bias = float(dev['best_bias'])

    torch.cuda.empty_cache()
    final_model, s1_history = train_stage1(train_df, tag='final', epochs=CFG['stage1_epochs'])
    final_model, s2_history, _ = train_stage2(
        final_model, train_df, tag='final', epochs=stage2_epochs, known_bias=dev_bias,
    )

    final_ckpt = torch.load(OUT_DIR / 'stage2_final_last.pt', map_location=DEVICE, weights_only=False)
    final_model.load_state_dict(final_ckpt['model'])

    ts = run_inference(final_model, make_loader(test_seen_df, eval_tf, CFG['eval_batch_size']))
    tu = run_inference(final_model, make_loader(test_unseen_df, eval_tf, CFG['eval_batch_size']))
    S_test, U_test = comp_scores(ts), comp_scores(tu)
    test_sweep = bias_sweep(S_test, ts['comp'], U_test, tu['comp'])

    def report(bias):
        s1 = topk_acc(S_test, ts['comp'], 1, bias)
        u1 = topk_acc(U_test, tu['comp'], 1, bias)
        return {
            'bias': round(float(bias), 3),
            'seen top-1': s1,
            'seen top-5': topk_acc(S_test, ts['comp'], 5, bias),
            'unseen top-1': u1,
            'unseen top-5': topk_acc(U_test, tu['comp'], 5, bias),
            'harmonic mean': harmonic_mean(s1, u1),
        }

    results = pd.DataFrame({
        'b = 0 (no calibration)': report(0.0),
        'b from dev validation (REPORTED)': report(dev_bias),
        'oracle b on test (upper bound)': report(test_sweep['best_bias']),
    }).T
    display(results.style.format({c: '{:.2%}' for c in results.columns if c != 'bias'}))
    print(f"seen/unseen AUC: {test_sweep['auc']:.4f}")

    head_table = pd.DataFrame({
        'test-seen': paper_protocol(ts),
        'test-unseen': paper_protocol(tu),
    }).T
    display(head_table.style.format('{:.2%}'))
    paper_hm = harmonic_mean(
        head_table.at['test-seen', 'crop+disease acc'],
        head_table.at['test-unseen', 'crop+disease acc'],
    )
    shortcut = shortcut_rate(tu)
    print(f'paper-protocol harmonic mean: {paper_hm:.2%}')
    print(f'shortcut rate: {shortcut:.2%}')

    plt.figure(figsize=(4.5, 4))
    plt.plot(test_sweep['seen'], test_sweep['novel'], '-')
    r = report(dev_bias)
    plt.scatter([r['seen top-1']], [r['unseen top-1']], c='r', zorder=3,
                label=f'dev b, H={r["harmonic mean"]:.3f}')
    plt.xlabel('test-seen top-1')
    plt.ylabel('test-unseen top-1')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title(f'Model 2 test bias sweep (AUC {test_sweep["auc"]:.3f})')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'test_bias_curve.png', dpi=150)
    plt.show()

    results_dict = {
        name: {k: (float(v) if isinstance(v, (float, int, np.floating, np.integer)) else v)
               for k, v in row.items()}
        for name, row in results.iterrows()
    }
    heads_dict = {
        split: {k: float(v) for k, v in row.items()}
        for split, row in head_table.iterrows()
    }
    metrics = dict(
        variant='model2_factorized_supcon',
        stage1_epochs=CFG['stage1_epochs'],
        stage2_epochs=stage2_epochs,
        dev_bias=dev_bias,
        results=results_dict,
        auc=float(test_sweep['auc']),
        heads=heads_dict,
        paper_protocol_hm=float(paper_hm),
        shortcut_rate=float(shortcut),
        data_fingerprint=data_fingerprint,
        cfg={k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()},
    )
    with open(OUT_DIR / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    print('saved:', sorted(p.name for p in OUT_DIR.glob('*')))

## Visual diagnostics

The final evaluation cell already saves the seen–unseen bias curve. This cell
adds the diagnostics needed to explain *why* Model 2 behaves the way it does:

- **Training curves** — Stage-1 total/crop/disease SupCon loss and Stage-2
  cross-entropy + head accuracy. Flat or diverging curves are a signal to check
  the learning rate or batch size.
- **Test bias sweep** — the full seen/unseen trade-off; the red point is the
  dev-selected operating point.
- **Confusion matrices** — row-normalized crop and disease matrices for
  test-seen and test-unseen. The diagonal is per-class recall; off-diagonal mass
  shows which classes are confused.
- **Composition-score distributions** — seen vs unseen maximum scores at $b=0$.
  More overlap means the novelty bias has to do more work.
- **Top-1 / top-5 bars** — a compact summary of the $b=0$ and dev-bias operating
  points.
- **t-SNE of backbone features** — a qualitative 2-D view of test-seen vs
  test-unseen features. Separation here is descriptive, not a metric; t-SNE
  distances should not be over-interpreted.

These plots are saved at 150 DPI under `runs/`. They are diagnostic aids; the
reportable numbers remain the dev-bias row in `metrics.json`.

In [1]:
if CFG['run_mode'] != 'final':
    print('Skipping visual diagnostics. Set CFG["run_mode"] = "final" after dev calibration.')
else:
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE

    ckpt_path = OUT_DIR / 'stage2_final_last.pt'
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Missing final checkpoint: {ckpt_path}. Run the final cell first.')

    if 'final_model' in globals() and 'ts' in globals() and 'tu' in globals():
        diag_model, ts, tu = final_model, ts, tu
    else:
        diag_ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        diag_model = build_model()
        diag_model.load_state_dict(diag_ckpt['model'])
        ts = run_inference(diag_model, make_loader(test_seen_df, eval_tf, CFG['eval_batch_size']))
        tu = run_inference(diag_model, make_loader(test_unseen_df, eval_tf, CFG['eval_batch_size']))

    dev = json.load(open(DEV_SUMMARY))
    dev_bias = float(dev['best_bias'])
    S, U = comp_scores(ts), comp_scores(tu)
    test_sweep = bias_sweep(S, ts['comp'], U, tu['comp'])

    def _report(bias):
        s1 = topk_acc(S, ts['comp'], 1, bias)
        u1 = topk_acc(U, tu['comp'], 1, bias)
        return dict(
            seen_top1=s1,
            unseen_top1=u1,
            seen_top5=topk_acc(S, ts['comp'], 5, bias),
            unseen_top5=topk_acc(U, tu['comp'], 5, bias),
            hm=harmonic_mean(s1, u1),
        )

    # 1) Training curves
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    s1_path, s2_path = OUT_DIR / 'history_stage1_final.csv', OUT_DIR / 'history_stage2_final.csv'
    if s1_path.exists():
        h1 = pd.read_csv(s1_path)
        axes[0].plot(h1['epoch'], h1['train_loss'], marker='o', label='total')
        axes[0].plot(h1['epoch'], h1['crop_supcon'], marker='s', label='crop SupCon')
        axes[0].plot(h1['epoch'], h1['disease_supcon'], marker='^', label='disease SupCon')
        axes[0].set_xlabel('epoch')
        axes[0].set_ylabel('loss')
        axes[0].set_title('Stage 1 — factorized SupCon')
        axes[0].legend(fontsize=8)
        axes[0].grid(alpha=0.25)
    if s2_path.exists():
        h2 = pd.read_csv(s2_path)
        ax2 = axes[1]
        ax2.plot(h2['epoch'], h2['train_loss'], marker='o', color='black', label='CE')
        ax2.set_xlabel('epoch')
        ax2.set_ylabel('cross-entropy')
        ax2.set_title('Stage 2 — linear probes')
        ax2.grid(alpha=0.25)
        ax3 = ax2.twinx()
        ax3.plot(h2['epoch'], h2['train_crop_acc'], '--', label='crop acc')
        ax3.plot(h2['epoch'], h2['train_disease_acc'], ':', label='disease acc')
        ax3.set_ylabel('train accuracy')
        ax3.set_ylim(0, 1)
        lines1, labels1 = ax2.get_legend_handles_labels()
        lines2, labels2 = ax3.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='best')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'training_curves_final.png', dpi=150)
    plt.show()
    plt.close(fig)

    # 2) Confusion matrices
    def _cm(y_true, y_pred, n):
        cm = np.zeros((n, n), dtype=np.float32)
        np.add.at(cm, (y_true.numpy(), y_pred.numpy()), 1.0)
        row = cm.sum(1, keepdims=True)
        row[row == 0] = 1.0
        return cm / row

    def _plot_cm(cm, title, path):
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
        ax.set_title(title)
        ax.set_xlabel('predicted class')
        ax.set_ylabel('true class')
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='row-normalized recall')
        fig.tight_layout()
        fig.savefig(path, dpi=150)
        plt.show()
        plt.close(fig)

    pc_s, pd_s = ts['crop_lp'].argmax(1), ts['dis_lp'].argmax(1)
    pc_u, pd_u = tu['crop_lp'].argmax(1), tu['dis_lp'].argmax(1)
    _plot_cm(_cm(ts['crop'], pc_s, NUM_CROPS), 'Crop confusion — test-seen', OUT_DIR / 'cm_crop_seen.png')
    _plot_cm(_cm(tu['crop'], pc_u, NUM_CROPS), 'Crop confusion — test-unseen', OUT_DIR / 'cm_crop_unseen.png')
    _plot_cm(_cm(ts['disease'], pd_s, NUM_DISEASES), 'Disease confusion — test-seen', OUT_DIR / 'cm_disease_seen.png')
    _plot_cm(_cm(tu['disease'], pd_u, NUM_DISEASES), 'Disease confusion — test-unseen', OUT_DIR / 'cm_disease_unseen.png')

    # 3) Score distributions
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(S.max(1).values.numpy(), bins=60, alpha=0.55, density=True, label='seen compositions')
    ax.hist(U.max(1).values.numpy(), bins=60, alpha=0.55, density=True, label='unseen compositions')
    ax.set_xlabel('max composition score at b=0')
    ax.set_ylabel('density')
    ax.set_title('Composition score overlap')
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'score_distribution.png', dpi=150)
    plt.show()
    plt.close(fig)

    # 4) Top-1 / top-5 bars
    r0, rb = _report(0.0), _report(dev_bias)
    metrics_names = ['seen top-1', 'unseen top-1', 'seen top-5', 'unseen top-5']
    vals0 = [r0['seen_top1'], r0['unseen_top1'], r0['seen_top5'], r0['unseen_top5']]
    valsb = [rb['seen_top1'], rb['unseen_top1'], rb['seen_top5'], rb['unseen_top5']]
    x = np.arange(len(metrics_names))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.2, vals0, width=0.4, label='b = 0')
    ax.bar(x + 0.2, valsb, width=0.4, label=f'dev b = {dev_bias:.2f}')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_names, rotation=15)
    ax.set_ylim(0, 1)
    ax.set_ylabel('accuracy')
    ax.set_title('Seen/unseen accuracy summary')
    ax.legend()
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'topk_summary.png', dpi=150)
    plt.show()
    plt.close(fig)

    # 5) t-SNE of backbone features (qualitative)
    n = min(600, len(ts['feat']), len(tu['feat']))
    rng = np.random.default_rng(42)
    idx_s = rng.choice(len(ts['feat']), n, replace=False)
    idx_u = rng.choice(len(tu['feat']), n, replace=False)
    X = np.vstack([ts['feat'][idx_s].numpy(), tu['feat'][idx_u].numpy()])
    X = (X - X.mean(0, keepdims=True)) / (X.std(0, keepdims=True) + 1e-6)
    emb = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto',
               random_state=42).fit_transform(X)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(emb[:n, 0], emb[:n, 1], s=8, alpha=0.55, label='test-seen')
    ax.scatter(emb[n:, 0], emb[n:, 1], s=8, alpha=0.55, label='test-unseen')
    ax.set_title('t-SNE of ViT features')
    ax.set_xlabel('dim 1')
    ax.set_ylabel('dim 2')
    ax.legend(markerscale=2)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'tsne_test_domains.png', dpi=150)
    plt.show()
    plt.close(fig)

    print('saved diagnostics:', sorted(p.name for p in OUT_DIR.glob('*.png')))

NameError: name 'CFG' is not defined

## Reporting notes

Use the **dev-bias** row as the headline result; the oracle row is only an
upper bound. Compare Model 2 against Baseline 2 ViT:

- Baseline 2 ViT final test: calibrated HM **58.06%**, AUC **0.5409**,
  unseen disease **9.32%**, shortcut **68.32%**.
- Model 2 target: improve unseen disease accuracy and reduce shortcut rate
  without collapsing crop accuracy or HM.
- Model 2 is **warm-started from the B2 ViT backbone** in the official protocol;
  B2's crop/disease heads are discarded. State this explicitly, because it is not
  a from-scratch comparison. If you fall back to ImageNet, say so as well.
- The B2 `metrics.json` `model` string incorrectly says ResNet-50; trust the
  checkpoint/config, which are ViT-B/16.

For the report, pair the final table with `runs/training_curves_final.png`,
`runs/test_bias_curve.png`, `runs/cm_disease_unseen.png`,
`runs/score_distribution.png`, and `runs/tsne_test_domains.png`. State clearly
that the dev bias is selected without test labels, while the oracle row is an
upper bound. The t-SNE plot is qualitative only. If Model 2 does not beat
Baseline 2, the honest conclusion is still useful: factorized SupCon may improve
representation structure without improving the final GZSL operating point under
this split and training budget.

Do not compare this dev HM directly with Baseline 1's dev HM; B1 used a different
backbone and protocol. Use the final test table for the report.